# colab_04_conformal_safe — safe vs pure-RL thresholds (CPU, FREE)

Gates: DECISION_01/02/03 + `human_check.csv` with `human_ok` filled (re-upload if needed).

Compares on CALIB only: heuristic tau vs learned tau (LTT) vs Mondrian per-bucket vs pure-RL argmax.

Freezes `thresholds.json` + DECISION_04.

In [ ]:
# 0. Setup — self-recovering: finds repo, unzips upload, or clones (public or private via token)
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = Path('/content/drive/MyDrive/Research/colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)
    print('Drive:', DRIVE)
except Exception as e:
    print('No Colab Drive (local run?):', e)
    DRIVE = Path('./colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)

def _find_backend():
    """Return BACKEND dir (containing src/gateway/service.py) or None."""
    cands = [Path('/content/code-mixed-gateway/backend'), Path('/content/Research/backend'),
             Path('/content/backend'), Path.cwd()/'backend', Path.cwd()/'code-mixed-gateway'/'backend']
    try:
        for p in Path('/content').iterdir():
            if p.is_dir():
                cands += [p, p/'backend']
    except Exception: pass
    for c in cands:
        if (c/'src'/'gateway'/'service.py').exists(): return c
    # deep search one level for upload_tmp style dirs
    try:
        for p in Path('/content').glob('*/**/service.py'):
            if p.parent.parent.name == 'gateway' and p.parent.name == 'gateway':
                be = p.parent.parent.parent  # .../src/gateway/service.py -> backend?
                # walk up to dir containing 'src'
                q = p.parent
                while q != Path('/content') and q.name != 'src': q = q.parent
                be = q.parent
                if (be/'src'/'gateway'/'service.py').exists(): return be
    except Exception: pass
    return None

def _try_unzip_uploads():
    roots = [Path('/content')]
    try: roots.append(DRIVE)
    except Exception: pass
    for root in roots:
        try: zips = list(root.glob('backend*.zip'))
        except Exception: continue
        for z in zips:
            dest = Path('/content/repo_upload')
            if (dest/'backend'/'src'/'gateway'/'service.py').exists(): return True
            print(f'unzipping {z} -> {dest} ...')
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(dest)
            # handle zip containing backend/ at top or nested one level
            if not (dest/'backend'/'src'/'gateway'/'service.py').exists():
                for sub in dest.iterdir():
                    if (sub/'backend'/'src'/'gateway'/'service.py').exists():
                        import shutil
                        shutil.move(str(sub/'backend'), str(dest/'backend_tmp'))
                        shutil.move(str(dest/'backend_tmp'), str(dest/'backend'))
                        break
            print('unzip done')
            return True
    return False

def _try_clone():
    dest = Path('/content/code-mixed-gateway')
    if dest.exists(): return dest.exists()
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''
    except Exception: token = os.environ.get('GITHUB_TOKEN','') or ''
    url = ('https://'+token+'@github.com/tusharsaharan/code-mixed-gateway.git') if token else 'https://github.com/tusharsaharan/code-mixed-gateway.git'
    print('cloning (token=%s) ...' % ('yes' if token else 'no'))
    r = subprocess.run(['git','clone',url,str(dest)], capture_output=True, text=True)
    print('clone rc=', r.returncode)
    if r.returncode != 0: print((r.stderr or '')[-500:])
    return dest.exists()

BACKEND = _find_backend()
if BACKEND is None and _try_unzip_uploads(): BACKEND = _find_backend()
if BACKEND is None and _try_clone(): BACKEND = _find_backend()
print('BACKEND =', BACKEND)
assert BACKEND is not None, ('REPO NOT FOUND. Fastest fix (2 min): on your PC zip the backend folder '
    '(Research/backend -> backend.zip), drag backend.zip into Colab Files panel (folder icon, left), then '
    'Cell > Run this cell again. Alternative: make the GitHub repo public, or add a GITHUB_TOKEN secret.')
REPO = BACKEND.parent
DATA = BACKEND/'data'
print('REPO =', REPO, '| DATA exists:', DATA.exists())

%pip install -q pydantic pydantic-settings numpy httpx 2>&1 | tail -1
src = str(BACKEND/'src')
if src not in sys.path: sys.path.insert(0, src)
import gateway
print('import gateway OK from', gateway.__file__)


In [ ]:
# 1. Build learned scores on calib: blend bandit + supervised (no test touch)
import json, numpy as np, pickle, torch
from gateway.modules.m3_conformal.conformal import ConformalCalibrator
from gateway.schemas import CalibSample
rows = [json.loads(l) for l in open(DRIVE/'calibration_real.jsonl', encoding='utf-8')]
by_id = {r['id']: r for r in rows}
splits = json.loads((DRIVE/'splits.json').read_text())
ca = splits['calib']
# heuristic scores
h = np.array([by_id[i]['nonconformity'] for i in ca])
y = np.array([0 if by_id[i]['cheap_success'] else 1 for i in ca])  # 1=fail
# supervised probs: reload logreg if dims match else fallback to heuristic
try:
    from sentence_transformers import SentenceTransformer
    from gateway.modules.m4_router.difficulty import DifficultyScorer
    lab = {json.loads(l)['id']: json.loads(l) for l in open(DRIVE/'groq_labels.jsonl', encoding='utf-8')}
    scorer = DifficultyScorer()
    enc = SentenceTransformer('all-MiniLM-L6-v2')
    E = enc.encode([lab[i]['text'] for i in ca], batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    import numpy as _np
    F = []
    for k, rid in enumerate(ca):
        f = scorer.features(lab[rid]['text'])
        F.append(_np.concatenate([_np.array([f.code_mix_ratio, f.entity_density, min(1,f.math_marker_count/3), min(1,f.char_count/400), len(lab[rid]['text'].split())/50], dtype=_np.float32), E[k]]))
    F = _np.stack(F)
    lr = pickle.load(open(DRIVE/'logreg.pkl','rb'))
    p_learned = lr.predict_proba(F)[:,1] if F.shape[1] == lr.coef_.shape[1] else h
except Exception as e:
    print('supervised reload fallback:', e); p_learned = h
print('learned mean', round(float(p_learned.mean()),4))

In [ ]:
# 2. LTT calibration: heuristic vs learned; global vs Mondrian. Pure-RL = reward-opt threshold (no guarantee).
def to_samples(scores, ids):
    return [CalibSample(id=i, nonconformity=round(float(s),6), cheap_success=bool(by_id[i]['cheap_success']), group=by_id[i]['group'], synthetic=False) for i, s in zip(ids, scores)]
out = {}
for name, sc in [('heuristic', h), ('learned', p_learned)]:
    cal = ConformalCalibrator(alpha=0.05, delta=0.05, method='fixed_sequence').calibrate(to_samples(sc, ca))
    out[name] = {'tau': round(cal.threshold,4), 'risk_hat': round(cal.risk_hat,4), 'risk_bound': round(cal.risk_bound,4),
        'cheap_share': round(float(sum(1 for v in sc if v < cal.threshold)/len(sc)),4)}
    print(name, out[name])
# Mondrian on learned
cal_m = ConformalCalibrator(alpha=0.05, delta=0.05).calibrate(to_samples(p_learned, ca))
out['mondrian'] = {'taus': {k: round(v,4) for k,v in cal_m.group_thresholds.items()}, 'risk_bound': round(cal_m.risk_bound,4), 'is_mondrian': cal_m.is_mondrian}
print(out['mondrian'])
# Pure-RL threshold: maximize saving - 2*fail on calib (lambda=2 frozen from NB03)
SAVE = 0.7; LAM = 2.0
best = max(((( (p_learned<t).astype(float)*(SAVE - LAM*y)).mean(), t) for t in [i/200 for i in range(201)]))
out['pure_rl'] = {'tau': round(float(best[1]),4), 'mean_reward': round(float(best[0]),4), 'guarantee': None}
print('pure_rl', out['pure_rl'])
(DRIVE/'thresholds.json').write_text(json.dumps({'alpha':0.05,'proxy':True,'arms':out}, indent=2))
(DRIVE/'DECISION_04.json').write_text(json.dumps({'stage':'04_conformal','frozen':'thresholds.json','note':'NB05 evaluates once on TEST; no retrain after'}, indent=2))
print('FROZEN thresholds.json')